# Reranking — cleanup pass over first-stage retrieval

First-stage retrievers (dense, BM25, hybrid) are tuned for recall. They throw a lot of barely-relevant chunks into the top-20. **Reranking** is a second stage that re-orders that candidate list with a stronger (slower, costlier) model.

Three reranker shapes:
1. **Cross-encoder** — joint (query, doc) encoder; the production default (BGE, Cohere Rerank).
2. **LLM-as-reranker** — zero-shot prompt; expensive but no training, easy to upgrade.
3. **Late-interaction / ColBERT-style** — covered in `02-indexing/06-colbert-late-interaction/`.

This notebook implements (1) as a deterministic stand-in (token-overlap score with an L2-norm penalty) and (2) by calling the LLM through the shared shim.

In [1]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'shared').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
if not (os.getenv('OPENAI_API_KEY') or os.getenv('ANTHROPIC_API_KEY')):
    os.environ.setdefault('LLM_CACHE_ONLY', '1')
print('LLM_CACHE_ONLY =', os.environ.get('LLM_CACHE_ONLY', '0'))


LLM_CACHE_ONLY = 1


In [2]:
import numpy as np
from shared.embedders import cosine_topk, hash_embed
from shared.llm import Message, complete
from shared.loaders import load_corpus, load_golden_qa

MODEL = 'openai/gpt-4o-mini'
NS = '01-rag/04-reranking'
DOCS = load_corpus()
QA = [q for q in load_golden_qa() if q.source_ids]
doc_lookup = {d.arxiv_id: d.title + '. ' + d.abstract for d in DOCS}
doc_texts = list(doc_lookup.values())
doc_ids = list(doc_lookup.keys())
doc_vecs = hash_embed(doc_texts, dims=256, seed=0)
print('docs:', len(DOCS), '  questions:', len(QA))

docs: 10   questions: 26


## First-stage candidates (dense top-5)
We over-retrieve k=5 with dense, then rerank down to top-1 for the metric.

In [3]:
def first_stage(q: str, k: int = 5) -> list[tuple[str, str]]:
    qv = hash_embed([q], dims=256, seed=0)[0]
    idx, _ = cosine_topk(qv, doc_vecs, k=k)
    return [(doc_ids[i], doc_texts[i]) for i in idx]

demo = first_stage(QA[0].question)
for doc_id, _ in demo:
    print(' ', doc_id)

  synth-001
  synth-008
  synth-004
  synth-010
  synth-002


## 1) Cross-encoder stand-in
We approximate a cross-encoder with a token-overlap score that *also* penalizes redundant context (so a doc that just repeats common words loses). Replace this function with `from sentence_transformers import CrossEncoder; CrossEncoder('BAAI/bge-reranker-base').predict([(q, d) for d in docs])` in production — the interface is identical.

In [4]:
import re
TOKEN_RE = re.compile(r'[A-Za-z0-9]+')
def tok(t: str) -> set[str]:
    return {w.lower() for w in TOKEN_RE.findall(t) if len(w) > 3}

def crossenc_score(q: str, d: str) -> float:
    qt, dt = tok(q), tok(d)
    if not qt or not dt:
        return 0.0
    overlap = len(qt & dt)
    return overlap / (len(dt) ** 0.5)  # length-normalized

def rerank_crossenc(q: str, cands: list[tuple[str, str]]) -> list[tuple[str, str]]:
    scored = sorted(cands, key=lambda c: -crossenc_score(q, c[1]))
    return scored

for doc_id, _ in rerank_crossenc(QA[0].question, first_stage(QA[0].question))[:3]:
    print(' ', doc_id)

  synth-001
  synth-008
  synth-002


## 2) LLM-as-reranker
Cheap when k is small. The prompt asks for just the bracketed id of the best candidate — no chain-of-thought, no scoring — which keeps it fast and easy to cache.

In [5]:
def rerank_llm(q: str, cands: list[tuple[str, str]]) -> list[tuple[str, str]]:
    ctx = '\n\n'.join(f'[{doc_id}] {text[:200]}...' for doc_id, text in cands)
    sys_p = ('You are a precise relevance ranker. Given a question and a list of '
             'candidate passages, return ONLY the bracketed id of the MOST relevant passage.')
    user_p = f'Question: {q}\n\nCandidates:\n{ctx}\n\nMost relevant id:'
    pick = complete(
        model=MODEL, namespace=NS,
        messages=[Message(role='system', content=sys_p), Message(role='user', content=user_p)],
    ).content.strip()
    pick = pick.strip('[]').split()[0] if pick else cands[0][0]
    # Move pick to the front; keep the rest of the order.
    out = [c for c in cands if c[0] == pick] + [c for c in cands if c[0] != pick]
    return out

for doc_id, _ in rerank_llm(QA[0].question, first_stage(QA[0].question))[:3]:
    print(' ', doc_id)

  synth-001
  synth-008
  synth-004


## Side-by-side eval — recall@1 before / after each reranker

In [6]:
def hit_at_1(top: list[tuple[str, str]], sources: list[str]) -> int:
    return int(top[0][0] in sources)

results = {'no_rerank': 0, 'crossenc': 0, 'llm': 0}
for item in QA:
    cands = first_stage(item.question, k=5)
    results['no_rerank'] += hit_at_1(cands, item.source_ids)
    results['crossenc'] += hit_at_1(rerank_crossenc(item.question, cands), item.source_ids)
    results['llm']      += hit_at_1(rerank_llm(item.question, cands), item.source_ids)
for k in results:
    results[k] = round(results[k] / len(QA), 4)
print(results)

{'no_rerank': 0.6923, 'crossenc': 0.9231, 'llm': 0.9615}


## Picking a reranker

| Reranker | Latency / k=20 | Cost | Recall lift |
|---|---|---|---|
| BGE base (cross-encoder) | ~30 ms | $0 (self-hosted) | strong |
| Cohere Rerank v3 | ~80 ms | ~$0.001 / call | strong |
| LLM-as-reranker (gpt-4o-mini) | ~400 ms | ~$0.0005 / call | moderate |
| Late-interaction (ColBERT v2) | ~100 ms | $0 (self-hosted) | strong |

Pick **cross-encoder first** for production. Reach for LLM-as-reranker when the rank logic itself is policy (e.g., "prefer recent docs", "prefer my employer's").